# PBx Score Leaderboard

**PBx** = maximum tokens-per-step (TPS) that achieves ≥ x% average score across all ParallelBench tasks.  
Higher PBx means the model can decode more tokens in parallel while maintaining quality.

Scores are computed via **linear interpolation** between adjacent (TPS, score) points.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display

from parallelbench.analysis.pb_score import DEFAULT_THRESHOLDS
from parallelbench.cli.analyze import (
    MODEL_DISPLAY_NAMES,
    METHOD_DISPLAY_NAMES,
    _build_leaderboard_records,
    _extract_rows_from_results,
    _find_latest_result_files,
)

RESULTS_DIR = Path("../results")

## Data Loading

Reuse `_find_latest_result_files` and `_extract_rows_from_results` from `parallelbench.cli.analyze`.

In [ ]:
latest_files = _find_latest_result_files(RESULTS_DIR)
all_rows = []
for f in latest_files:
    try:
        all_rows.extend(_extract_rows_from_results(f))
    except Exception as e:
        print(f"Warning: skipping {f}: {e}")

print(f"Loaded {len(all_rows)} rows from {len(latest_files)} result files")
print(f"Models: {sorted(set(r['model'] for r in all_rows))}")
print(f"Methods: {sorted(set(r['unmasking'] for r in all_rows))}")

## PBx Leaderboard — Per Model × Method

Reuse `_build_leaderboard_records` from `parallelbench.cli.analyze` to compute PBx scores per (model, method) group.

In [ ]:
records = _build_leaderboard_records(all_rows)

# Convert to DataFrame with display names
leaderboard = pd.DataFrame(
    [
        {
            "Model": MODEL_DISPLAY_NAMES.get(r["model"], r["model"]),
            "Method": METHOD_DISPLAY_NAMES.get(r["method"], r["method"]),
            **{k: v for k, v in r.items() if k.startswith("PB")},
        }
        for r in records
    ]
)

# Sort by PB80, PB70, PB60, PB90 descending (matching CLI)
leaderboard_sorted = leaderboard.sort_values(
    ["PB80", "PB70", "PB60", "PB90"], ascending=False, na_position="last"
).reset_index(drop=True)
leaderboard_sorted.index = leaderboard_sorted.index + 1
leaderboard_sorted.index.name = "Rank"

# Display thresholds (exclude PB90)
display_thresholds = [t for t in DEFAULT_THRESHOLDS if t != 90]

# Format for display
display_df = leaderboard_sorted[
    ["Model", "Method", *[f"PB{t}" for t in display_thresholds]]
].copy()
for col in [f"PB{t}" for t in display_thresholds]:
    if col in display_df.columns:
        display_df[col] = display_df[col].apply(
            lambda x: f"{x:.1f}" if pd.notna(x) else "-"
        )

display(display_df)

## Best Method per Model

For each model, show only the best-performing unmasking method (by PB80).

In [ ]:
# Best method per model (by PB80, PB70, PB60, PB90)
best_per_model = (
    leaderboard.sort_values(
        ["PB80", "PB70", "PB60", "PB90"], ascending=False, na_position="last"
    )
    .drop_duplicates(subset=["Model"], keep="first")
    .sort_values(["PB80", "PB70", "PB60", "PB90"], ascending=False, na_position="last")
    .reset_index(drop=True)
)
best_per_model.index = best_per_model.index + 1
best_per_model.index.name = "Rank"

display_thresholds = [t for t in DEFAULT_THRESHOLDS if t != 90]

best_display = best_per_model[
    ["Model", "Method", *[f"PB{t}" for t in display_thresholds]]
].copy()
for col in [f"PB{t}" for t in display_thresholds]:
    if col in best_display.columns:
        best_display[col] = best_display[col].apply(
            lambda x: f"{x:.1f}" if pd.notna(x) else "-"
        )

display(best_display)

## PBx Bar Chart

Visual comparison of PB80 scores across all (model, method) combinations.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update(
    {
        "figure.dpi": 150,
        "font.size": 10,
        "axes.titlesize": 13,
        "axes.labelsize": 11,
    }
)

# Method colors
METHOD_COLORS = {
    "Confidence Top-K": "#2563eb",
    "Confidence Threshold": "#7c3aed",
    "Confidence Factor": "#db2777",
    "Entropy Top-K": "#059669",
    "Top-K Margin": "#d97706",
    "Random": "#6b7280",
    "Left-to-Right": "#dc2626",
}

# Prepare data: filter out rows where PB80 is NaN, sort ascending for horizontal bar
plot_df = leaderboard_sorted.dropna(subset=["PB80"]).copy()
# Use raw numeric values (not the formatted display_df)
plot_df = (
    leaderboard.dropna(subset=["PB80"])
    .sort_values("PB80", ascending=True)
    .reset_index(drop=True)
)
plot_df["label"] = plot_df["Model"] + " / " + plot_df["Method"]

fig, ax = plt.subplots(figsize=(10, max(4, len(plot_df) * 0.4)))

colors = [METHOD_COLORS.get(m, "#94a3b8") for m in plot_df["Method"]]
bars = ax.barh(
    plot_df["label"], plot_df["PB80"], color=colors, edgecolor="white", linewidth=0.5
)

# Value labels
for bar, val in zip(bars, plot_df["PB80"]):
    ax.text(
        bar.get_width() + 0.3,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.1f}",
        va="center",
        fontsize=9,
    )

ax.set_xlabel("PB80 (TPS achieving ≥ 80% avg score)")
ax.set_title("PBx Leaderboard — PB80")
ax.set_xlim(0, plot_df["PB80"].max() * 1.15)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## Grouped Bar Chart — PBx Thresholds per Model (Best Method)

In [ ]:
import numpy as np

display_thresholds = [t for t in DEFAULT_THRESHOLDS if t != 90]
threshold_cols = [f"PB{t}" for t in display_thresholds]
threshold_colors = {
    "PB80": "#2563eb",
    "PB70": "#059669",
    "PB60": "#d97706",
}

plot_best = best_per_model.sort_values(
    "PB80", ascending=False, na_position="last"
).reset_index(drop=True)
models = plot_best["Model"] + "\n(" + plot_best["Method"] + ")"

x = np.arange(len(models))
width = 0.25

fig, ax = plt.subplots(figsize=(max(8, len(models) * 2), 5))

for i, col in enumerate(threshold_cols):
    values = plot_best[col].fillna(0)
    bars = ax.bar(
        x + i * width,
        values,
        width,
        label=col,
        color=threshold_colors.get(col, "#94a3b8"),
    )
    for bar, val, raw in zip(bars, values, plot_best[col]):
        if pd.notna(raw) and val > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.3,
                f"{val:.1f}",
                ha="center",
                va="bottom",
                fontsize=8,
            )

ax.set_xticks(x + width * 1.0)
ax.set_xticklabels(models, fontsize=9)
ax.set_ylabel("Tokens per Step (TPS)")
ax.set_title("PBx Scores — Best Method per Model")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()